# Petals to the Metal — 从零实现 GoogLeNet（PyTorch + TPU）

本 Notebook 是一份**完整的 PyTorch 入门参考**，涵盖：
1. 用 TensorFlow 读取 TFRecord 竞赛数据
2. 从零手写 GoogLeNet (Inception v1) + BatchNorm
3. 在 Kaggle TPU 上训练 104 类花卉分类模型
4. 推理测试集并生成 `submission.csv`

**适合人群**：想学习如何用 PyTorch 从零搭建 CNN、跑 TPU 训练、打 Kaggle 比赛的同学。

## 1. 环境准备

Kaggle 环境预装了 PyTorch 和 TensorFlow。我们需要额外安装 `torch_xla` 来使用 TPU。

In [ ]:
!pip install -q torch_xla cloud-tpu-client

## 2. 导入库 & TPU 初始化

In [ ]:
import io, time
from pathlib import Path

import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ── TPU 检测 ─────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    import torch_xla.distributed.xla_multiprocessing as xmp

    devices = xm.get_xla_supported_devices()
    if devices:
        device = xm.xla_device()
        tpu_available = True
        print(f'TPU available: {len(devices)} cores')
    else:
        raise RuntimeError('No TPU devices')
except Exception:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tpu_available = False
    print(f'TPU not available, using: {device}')
    if device.type == 'cuda':
        print(f'  GPU: {torch.cuda.get_device_name(0)}')

# 全局常量
NUM_CLASSES = 104
IMAGE_SIZE  = 224
BATCH_SIZE  = 128            # 单 core 的 batch
EPOCHS      = 60
LR          = 1e-3

if tpu_available:
    BATCH_SIZE = 64           # TPU 显存较小，降低 batch
    print(f'  TPU batch size adjusted to {BATCH_SIZE}')

## 3. 数据集：TensorFlow 读取 TFRecord

Kaggle 官方数据是 TFRecord 格式（JPEG 编码的图片 + 标签）。我们用 TF 解析，PyTorch Dataset 封装。

**为什么用 TensorFlow 而不是 `tfrecord` 包？**
—— Kaggle 环境预装了 TF，零额外依赖。

In [ ]:
class PetalsDataset(Dataset):
    """用 TensorFlow 解析 TFRecord，封装为 PyTorch Dataset。

    数据目录结构::

        data/
        └── tfrecords-jpeg-224x224/
            ├── train/   (16 个 .tfrec 文件)
            ├── val/     (16 个 .tfrec 文件)
            └── test/    (16 个 .tfrec 文件)
    """

    def __init__(self, data_dir, image_size=224, split='train', transform=None):
        if split not in ('train', 'val', 'test'):
            raise ValueError(f"split must be 'train', 'val', or 'test', got '{split}'")
        self.split = split
        self.transform = transform
        self.samples = []

        # 1. 找到所有 .tfrec 文件
        tfrecord_dir = Path(data_dir) / f'tfrecords-jpeg-{image_size}x{image_size}' / split
        tfrecord_paths = sorted(tfrecord_dir.glob('*.tfrec'))
        if not tfrecord_paths:
            raise FileNotFoundError(f"No .tfrec files in '{tfrecord_dir}'")

        # 2. 定义 TFRecord schema（test 集没有 class 字段）
        if split == 'test':
            feature_desc = {
                'image': tf.io.FixedLenFeature([], tf.string),
                'id':    tf.io.FixedLenFeature([], tf.string),
            }
        else:
            feature_desc = {
                'image': tf.io.FixedLenFeature([], tf.string),
                'class': tf.io.FixedLenFeature([], tf.int64),
            }

        # 3. 读取全部记录到内存
        raw_ds = tf.data.TFRecordDataset([str(p) for p in tfrecord_paths])
        for record in raw_ds:
            parsed = tf.io.parse_single_example(record, feature_desc)
            sample = {'image': parsed['image'].numpy()}         # JPEG 字节
            if split == 'test':
                sample['id'] = parsed['id'].numpy()
            else:
                sample['class'] = parsed['class'].numpy()      # int64 标签
            self.samples.append(sample)
        print(f'  {split}: {len(self.samples)} 张图片')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        r = self.samples[idx]
        # JPEG byte → RGB PIL Image
        img = Image.open(io.BytesIO(r['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.split == 'test':
            img_id = r['id'].decode('utf-8') if isinstance(r['id'], bytes) else r['id']
            return img, img_id
        return img, int(r['class'])

## 4. 模型：GoogLeNet (Inception v1)

GoogLeNet 的核心思想是 **Inception 模块** —— 用多条并行的不同尺度卷积同时提取特征，再在通道维拼接。

**架构概览**：

```
Input 3×224×224
  │
  ▼ Stem (Conv+Pool)
  │
  ▼ Inception 3a, 3b  →  28×28
  │
  ▼ MaxPool           →  14×14
  │
  ▼ Inception 4a~4e  →  14×14
  │
  ▼ MaxPool           →   7×7
  │
  ▼ Inception 5a, 5b  →   7×7
  │
  ▼ GlobalAvgPool     →   1×1
  │
  ▼ FC(1024)→FC(104)  →  输出 104 类
```

每个卷积层后面都跟了 **BatchNorm + ReLU**，防止梯度消失。

In [ ]:
class Inception(nn.Module):
    """Inception 模块 —— 四条路径并行，通道拼接。

    路径 1: 1×1 卷积
    路径 2: 1×1 → 3×3 卷积
    路径 3: 1×1 → 5×5 卷积
    路径 4: 3×3 MaxPool → 1×1 卷积

    四个输出在通道维拼接。
    """

    def __init__(self, in_channels, c1, c2, c3, c4):
        super().__init__()
        # 路径 1
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=1),
            nn.BatchNorm2d(c1), nn.ReLU(inplace=True))

        # 路径 2
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c2[0], kernel_size=1),
            nn.BatchNorm2d(c2[0]), nn.ReLU(inplace=True),
            nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1),
            nn.BatchNorm2d(c2[1]), nn.ReLU(inplace=True))

        # 路径 3
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c3[0], kernel_size=1),
            nn.BatchNorm2d(c3[0]), nn.ReLU(inplace=True),
            nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2),
            nn.BatchNorm2d(c3[1]), nn.ReLU(inplace=True))

        # 路径 4
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, c4, kernel_size=1),
            nn.BatchNorm2d(c4), nn.ReLU(inplace=True))

    def forward(self, x):
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x),
        ], dim=1)


class GoogLeNet(nn.Module):
    """GoogLeNet (Inception v1)，约 710 万参数。

    Args:
        num_classes: 分类数（104 类花卉）
        dropout:     全连接层 dropout 率
    """

    def __init__(self, num_classes=104, dropout=0.5):
        super().__init__()

        # ── Stem：两个卷积 + 池化，快速降采样 ──
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            nn.Conv2d(64, 64, kernel_size=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        # ── Inception 3a, 3b（28×28 → 14×14）──
        self.inception3a = Inception(192, 64,  (96, 128),  (16, 32), 32)
        self.inception3b = Inception(256, 128, (128, 192), (32, 96), 64)
        self.pool3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ── Inception 4a~4e（14×14 → 7×7）──
        self.inception4a = Inception(480, 192, (96, 208),  (16, 48),  64)
        self.inception4b = Inception(512, 160, (112, 224), (24, 64),  64)
        self.inception4c = Inception(512, 128, (128, 256), (24, 64),  64)
        self.inception4d = Inception(512, 112, (144, 288), (32, 64),  64)
        self.inception4e = Inception(528, 256, (160, 320), (32, 128), 128)
        self.pool4 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ── Inception 5a, 5b（7×7）──
        self.inception5a = Inception(832, 256, (160, 320), (32, 128), 128)
        self.inception5b = Inception(832, 384, (192, 384), (48, 128), 128)

        # ── 分类头 ──
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.pool3(x)

        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.pool4(x)

        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = self.classifier(x)
        return x

## 5. 训练引擎

包含：累加器、准确率计算、训练一轮、验证、完整训练循环。

TPU 训练的关键差异：
- `xm.optimizer_step(optimizer)` 代替 `optimizer.step()`
- `MpDeviceLoader` 包装 DataLoader 以预取数据到 TPU

In [ ]:
class Accumulator:
    """累加器 —— 方便在多个 batch 上累加 loss 和准确率。"""
    def __init__(self, n):
        self.data = [0.0] * n
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    def __getitem__(self, i):
        return self.data[i]


def accuracy(y_hat, y):
    """计算预测正确的数量。"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(dim=1)
    return float((y_hat.type(y.dtype) == y).type(y.dtype).sum())


def evaluate(net, data_iter, eval_device):
    """在验证集上评估准确率。"""
    net.eval()
    metric = Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(eval_device), y.to(eval_device)
            y_hat = net(X)
            metric.add(accuracy(y_hat, y), y.numel())
    return metric[0] / metric[1]


def train_one_epoch(net, loader, loss_fn, optimizer, train_device, scaler=None):
    """训练一个 epoch。TPU 时用 ``xm.optimizer_step``。

    返回 (avg_loss, accuracy)。
    """
    net.train()
    metric = Accumulator(3)

    for X, y in loader:
        X, y = X.to(train_device), y.to(train_device)
        optimizer.zero_grad()

        y_hat = net(X)
        l = loss_fn(y_hat, y)
        l.mean().backward()

        if tpu_available:
            xm.optimizer_step(optimizer)           # ← TPU 专用
        else:
            optimizer.step()                       # ← GPU / CPU

        metric.add(l.sum().item() * y.numel(), accuracy(y_hat, y), y.numel())
    return metric[0] / metric[2], metric[1] / metric[2]


def train_model(net, train_iter, val_iter, num_epochs, lr):
    """完整训练流程：TPU / GPU / CPU 自适应。

    返回: (train_losses, train_accs, val_accs, best_val_acc)
    """
    net.to(device)

    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    loss_fn = nn.CrossEntropyLoss()

    # TPU 需要用 MpDeviceLoader 预取数据
    if tpu_available:
        train_loader = pl.MpDeviceLoader(train_iter, device)
        val_loader   = pl.MpDeviceLoader(val_iter, device)
    else:
        train_loader = train_iter
        val_loader   = val_iter

    train_losses, train_accs, val_accs = [], [], []
    best_acc = 0.0
    t0_total = time.time()

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        train_l, train_acc = train_one_epoch(net, train_loader, loss_fn, optimizer, device)
        val_acc = evaluate(net, val_loader, device)
        scheduler.step()

        train_losses.append(train_l)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(net.state_dict(), 'best_model.pth')

        elapsed = time.time() - t0
        eta = (elapsed / epoch) * (num_epochs - epoch)
        eta_str = f' | ETA: {eta/60:.0f}min' if eta > 60 else f' | ETA: {eta:.0f}s'

        if epoch % 5 == 0 or epoch == 1:
            print(f'epoch {epoch:3d}/{num_epochs} | '
                  f'train loss {train_l:.4f}, train acc {train_acc:.4f}, '
                  f'val acc {val_acc:.4f} | '
                  f'{elapsed:.0f}s/epoch{eta_str}')

    total = time.time() - t0_total
    print(f'\n训练完成！总用时: {total/60:.1f}min  |  最佳 val acc: {best_acc:.4f}')
    return train_losses, train_accs, val_accs, best_acc

## 6. 数据增强 & 加载

In [ ]:
DATA_DIR = '/kaggle/input/tpu-getting-started'

# ── 训练 transform（带增强）──
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=25),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── 验证 transform（不做增强）──
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('正在加载数据...')
train_ds = PetalsDataset(DATA_DIR, IMAGE_SIZE, 'train', train_tf)
val_ds   = PetalsDataset(DATA_DIR, IMAGE_SIZE, 'val',   val_tf)
print(f'训练集: {len(train_ds)} 张  |  验证集: {len(val_ds)} 张')

# num_workers=0 避免 Windows 多进程问题
train_iter = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_iter   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 7. 训练！

In [ ]:
model = GoogLeNet(num_classes=NUM_CLASSES, dropout=0.5)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')

# 下面这行开始训练。预计 TPU v3-8 上约 30~50 分钟
train_losses, train_accs, val_accs, best_acc = train_model(
    model, train_iter, val_iter,
    num_epochs=EPOCHS, lr=LR,
)

## 8. 推理 & 生成提交文件

In [ ]:
# 加载最佳模型权重
model.load_state_dict(torch.load('best_model.pth',
                        map_location=torch.device('cpu')))
model.to(device).eval()

# 测试集（不做增强）
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('正在加载测试集...')
test_ds = PetalsDataset(DATA_DIR, IMAGE_SIZE, 'test', test_tf)
test_iter = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
print(f'测试集: {len(test_ds)} 张')

ids_all, preds_all = [], []
with torch.no_grad():
    for imgs, img_ids in test_iter:
        logits = model(imgs.to(device))
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())
        ids_all.extend(img_ids)

print(f'预测完成: {len(ids_all)} 条')

# 写入 submission.csv
with open('/kaggle/working/submission.csv', 'w') as f:
    f.write('id,label\n')
    for img_id, pred in zip(ids_all, preds_all):
        f.write(f'{img_id},{pred}\n')
print('/kaggle/working/submission.csv 已保存！')

## 9. 总结

### 关键设计决策

| 决策 | 选择 | 原因 |
|------|------|------|
| 模型 | GoogLeNet (7.1M 参数) | 轻量、适合入门学习 |
| 归一化 | BatchNorm（每层卷积后） | 防止梯度消失，加速收敛 |
| 优化器 | AdamW + CosineAnnealing | 自适应学习率 + 衰减 |
| 数据增强 | RandomCrop + HFlip + Rotation | 抑制过拟合 |
| ColorJitter | **不用** | 花卉分类颜色是关键特征 |
| TPU 数据加载 | MpDeviceLoader | TPU 需要异步预取 |
| TPU 优化器步进 | xm.optimizer_step() | TPU 多核梯度同步 |

### 预期效果

- 验证准确率：60 轮约 **73~78%**（从零训练）
- 比赛前排约 95%+（预训练 EfficientNet / ConvNeXt）

### 改进方向

- 加载 ImageNet 预训练权重微调（准确率 +15~20%）
- 加 MixUp / CutMix 增广（减小过拟合）
- 换更大的 backbone（ConvNeXt, Swin Transformer）